# ਪਾਠ 04 - ਟੂਲ ਉਸੇ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਮਾਈਕ੍ਰੋਸੌਫਟ ਐਜੰਟ ਫ੍ਰੇਮਵਰਕ (Python) ਦੀ ਵਰਤੋਂ ਕਰਦੇ ਹੋਏ AI ਏਜੰਟ ਲਈ **ਟੂਲ ਉਸੇ** ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ ਬਾਰੇ ਸਿੱਖੋਗੇ। ਅਸੀਂ ਕਵਰ ਕਰਾਂਗੇ:

- `@tool` ਡੈੱਕੋਰੇਟਰ ਅਤੇ ਟਾਈਪਡ ਪੈਰਾਮੀਟਰਾਂ ਨਾਲ ਫੰਕਸ਼ਨ ਟੂਲ تعريف ਕਰਨਾ
- ਟੂਲ ਸਕੀਮਾਂ ਪ੍ਰਦਾਨ ਕਰਨਾ ਤਾਂ ਜੋ ਮਾਡਲ ਨੂੰ ਪਤਾ ਚੱਲੇ ਕਿ ਹਰ ਟੂਲ ਕੀ ਕਰਦਾ ਹੈ
- `approval_mode` ਨਾਲ ਟੂਲ ਚਲਾਉਣ ਤੇ ਕੰਟਰੋਲ ਕਰਨਾ
- Pydantic ਮਾਡਲਾਂ ਅਤੇ `response_format` ਰਾਹੀਂ **ਸੰਰਚਿਤ ਆਉਟਪੁੱਟ** ਵਾਪਸ ਕਰਨਾ

ਸਟੇਨੀਓ ਹੈ ਇੱਕ **ਯਾਤਰਾ ਬੁਕਿੰਗ ਏਜੰਟ** ਜੋ ਟਿਕਾਣਿਆਂ ਦੀ ਜਾਂਚ ਕਰ ਸਕਦਾ ਹੈ, ਉਪਲਬਧਤਾ ਤਸਦੀਕ ਕਰ ਸਕਦਾ ਹੈ ਅਤੇ ਉਡਾਣ ਦੀ ਜਾਣਕਾਰੀ ਪ੍ਰਾਪਤ ਕਰ ਸਕਦਾ ਹੈ।


## ਸੈਟਅੱਪ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Defining Tools with the @tool Decorator

The `@tool` decorator turns a plain Python function into a tool that an agent can call.
Key points:

- The **docstring** becomes the tool description the model sees.
- **Type annotations** (including `Annotated` with descriptions) define the tool schema.
- `approval_mode` controls whether the user must approve each call before it executes.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## ਕਈ ਟੂਲਾਂ ਨਾਲ ਏਜੰਟ ਬਣਾਉਣਾ

ਮਾਡਲ ਨੂੰ ਉਹਨਾਂ ਵਿੱਚੋਂ ਜਿਸ ਦੀ ਲੋੜ ਹੋਵੇ ਉਸਨੂੰ ਕਾਲ ਕਰਨ ਦੇ ਲਈ ਸਾਰੇ ਤਿੰਨ ਟੂਲ ਕਲਾਇਂਟ ਨੂੰ ਦੇ ਦਿਓ ਤਾਂ ਜੋ ਉਹ ਉਪਭੋਗਤਾ ਦੇ ਸਵਾਲ ਦਾ ਜਵਾਬ ਦੇ ਸਕੇ।


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## ਟੂਲਾਂ ਨਾਲ ਬਣਾਈ ਗਈ ਬਣਤਰਬੱਧ ਆઉਟਪੁੱਟ

`response_format` ਨੂੰ ਇਕ ਪਾਇਡੈਂਟਿਕ ਮਾਡਲ ਤੇ ਸੈੱਟ ਕਰਕੇ, ਏਜੰਟ ਨੂੰ ਆਜ਼ਾਦ ਲਿਖਤ ਦੀ ਥਾਂ ਇੱਕ ਵਧੀਆ ਟਾਈਪ ਕੀਤਾ ਗਿਆ JSON ਵਸਤੂ ਵਾਪਸ ਕਰਨ ਲਈ ਮਜ਼ਬੂਰ ਕੀਤਾ ਜਾਂਦਾ ਹੈ। ਇਹ ਉਸ ਵੇਲੇ ਲਾਭਦਾਇਕ ਹੁੰਦਾ ਹੈ ਜਦੋਂ ਹੇਠਲੇ ਕੋਡ ਨੂੰ ਨਤੀਜੇ ਨੂੰ ਪ੍ਰੋਗ੍ਰਾਮੈਟਿਕ ਤਰੀਕੇ ਨਾਲ ਖਪਤ ਕਰਨ ਦੀ ਲੋੜ ਹੁੰਦੀ ਹੈ।


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## ਟੂਲ ਮਨਜ਼ੂਰੀ ਪੈਟਰਨ

`@tool` 'ਤੇ `approval_mode` ਪੈਰामीਟਰ ਇਸ ਗੱਲ ਨੂੰ ਨਿਯੰਤਰਿਤ ਕਰਦਾ ਹੈ ਕਿ ਕੀ ਟੂਲ ਕਾਲਾਂ ਨੂੰ ਚਲਾਉਣ ਤੋਂ ਪਹਿਲਾਂ ਮਨੁੱਖੀ ਮਨਜ਼ੂਰੀ ਦੀ ਲੋੜ ਹੈ:

| ਮੋਡ | ਵਰਤਾਰਾ |
|---|---|
| `"never_require"` | ਟੂਲ ਆਪੇ ਆਪ ਚੱਲਦਾ ਹੈ — ਕੋਈ ਯੂਜ਼ਰ ਪੁਸ਼ਟੀ ਕਰਨ ਦੀ ਲੋੜ ਨਹੀਂ। |
| `"always_require"` | ਹਰ ਕਾਲ ਨੂੰ ਚਲਾਉਣ ਤੋਂ ਪਹਿਲਾਂ ਯੂਜ਼ਰ ਵੱਲੋਂ ਮਨਜ਼ੂਰ ਕੀਤਾ ਜਾਣਾ ਜ਼ਰੂਰੀ ਹੈ। |

ਜਿਨ੍ਹਾਂ ਟੂਲਾਂ ਦੇ ਸਾਈਡ-ਇਫੈਕਟ ਹੁੰਦੇ ਹਨ (ਜਿਵੇਂ ਕਿ ਉਡਾਣ ਬੁਕ ਕਰਨਾ, ਕਰੈਡਿਟ ਕਾਰਡ ਚਾਰਜ ਕਰਨਾ), ਓਹਨਾਂ ਲਈ `"always_require"` ਵਰਤੋਂ ਤਾਂ ਜੋ ਇੱਕ ਮਨੁੱਖ ਸਦਾ ਇਸ ਚੱਕਰ ਵਿੱਚ ਰਹੇ। 


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## ਸਾਰ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਸਿੱਖਿਆ ਕਿ:

1. **ਸੰਦਾਂ ਨੂੰ ਪਰਿਭਾਸ਼ਿਤ ਕਰੋ** `@tool` ਡੇਕੋਰੇਟਰ ਦੀ ਵਰਤੋਂ ਕਰਕੇ, ਜਿਸ ਵਿੱਚ ਟਾਇਪ ਕੀਤੇ ਪੈਰਾਮੀਟਰ ਅਤੇ ਦਸਤਾਵੇਜ਼ ਸੂਤਰ ਹੁੰਦੇ ਹਨ ਜੋ ਕਰਨ ਵਾਲੇ ਸੰਦ ਦੀ ਸਕੀਮਾ ਹੁੰਦੇ ਹਨ।
2. **ਕਈ ਸੰਦਾਂ ਨੂੰ ਮਿਲਾ ਕੇ ਰਚਨਾ ਕਰੋ** ਤਾਂ ਜੋ ਏਜੰਟ ਉਨ੍ਹਾਂ ਨੂੰ ਕ੍ਰਮ ਵਿੱਚ ਕਾਲ ਕਰਕੇ ਜਟਿਲ ਸਵਾਲਾਂ ਦੇ ਜਵਾਬ ਦੇ ਸਕੇ।
3. **ਸੰਰਚਿਤ ਨਤੀਜਾ ਵਾਪਸ ਕਰੋ** ਪਾਇਡੈਂਟਿਕ ਮਾਡਲ ਨੂੰ `response_format` ਵਜੋਂ ਪਾਸ ਕਰਕੇ।
4. **ਸੰਦ ਦੀ ਮਨਜ਼ੂਰੀ ਨੂੰ ਨਿਯੰਤਰਿਤ ਕਰੋ** `approval_mode` ਨਾਲ ਤਾਂ ਜੋ ਸੰਵੇਦਨਸ਼ੀਲ ਕਾਰਵਾਈਆਂ ਲਈ ਮਨੁੱਖੀ ਹਸਤਖੇਪ ਜਾਰੀ ਰਹੇ।

ਇਹ ਨਮੂਨੇ ਭਰੋਸੇਯੋਗ, ਉਤਪਾਦਨ-ਤਿਆਰ ਏਜੰਟ ਬਣਾਉਣ ਦੀ ਬੁਨਿਆਦ ਪੇਸ਼ ਕਰਦੇ ਹਨ ਜੋ ਬਾਹਰੀ ਪ੍ਰਣਾਲੀਆਂ ਨਾਲ ਸੁਰੱਖਿਅਤ ਢੰਗ ਨਾਲ ਸੰਵਾਦ ਕਰ ਸਕਦੇ ਹਨ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
